# ScreamingFace — Bring Your Own Keys: IF-Eval Guide

**Audience:** researchers already familiar with LLM evaluation who want to run IF-Eval
locally with their own API credentials and understand how the ScreamingFace engine works
under the hood.

This notebook covers:

1. Booting a local ScreamingFace engine inside this Colab VM
2. Connecting one of four provider options (OpenRouter, OpenAI, Anthropic, Hugging Face)
3. How IF-Eval is implemented in the engine — verifier, pinning, scoring protocol
4. The caching layer and what it means for tokenomics
5. The three candidate primitives — `sf.Model`, `sf.Fusion`, `sf.Pipeline` — and how
   they compose into corrective patterns
6. Running an evaluation and reading scores, token counts, and cost
7. URL4 — the compiled expression that pins a recipe for exact reproducibility

A companion notebook (`ScreamingFace_Platform.ipynb`) covers the hosted engine path,
which requires no local install but uses shared OpenRouter credits and a Google login.

---
## 1 · Install the runtime and boot the local engine

In [ ]:
%pip install -q "screamingface[runtime,notebook]"

The next cell downloads the pinned IF-Eval assets (dataset + vendored verifier) and starts
three background services:

| Service | Port | Role |
|---------|------|------|
| Gateway | 9105 | Provider credential store, LLM dispatch, request cache |
| Scoreboard | 9106 | Leaderboard read/write |
| Engine | 9108 | Benchmark protocol execution, URL4 compilation |

`screamingface status` should show all three healthy. If any fail, run
`!screamingface logs` to inspect startup output.

In [ ]:
!screamingface prepare ifeval
!screamingface up
!screamingface status

---
## 2 · Configure the client and connect a provider

`sf.configure(...)` points the client at the engine running in this VM.
Then pick exactly one of the four provider cells below and run it.

In [ ]:
import screamingface as sf

sf.configure(
    engine_url="http://127.0.0.1:9108",
    scoreboard_url="http://127.0.0.1:9106",
)

### Provider options

Run **exactly one** of the four cells below. The key is stored encrypted in the local
Gateway (AES-256-GCM); the notebook itself never retains it.

| Option | Best for | Notes |
|--------|----------|-------|
| **A — OpenRouter** | Everything, including Gemini | Full model catalog; recommended as primary path |
| **B — OpenAI direct** | GPT models, no intermediary | Direct API; Gemini not available this way |
| **C — Anthropic direct** | Claude models, no intermediary | Direct API |
| **D — Hugging Face** | Open-source / self-hosted models | HF Inference API |

> **Gemini models** (including `gemini-3.1-flash` used in the tokenomics paper) are only
> accessible via OpenRouter in BYOK mode — use option A.

In [ ]:
# Option A — OpenRouter (recommended; covers GPT, Claude, Gemini, and open-source models)
# Get a key at https://openrouter.ai/keys
OPENROUTER_KEY = ""  # paste sk-or-...

sf.connect("openrouter", api_key=OPENROUTER_KEY)

In [ ]:
# Option B — OpenAI direct (GPT models only, no Gemini)
# Get a key at https://platform.openai.com/api-keys
OPENAI_KEY = ""  # paste sk-...

sf.connect("openai", api_key=OPENAI_KEY)

In [ ]:
# Option C — Anthropic direct (Claude models only)
# Get a key at https://console.anthropic.com/settings/keys
ANTHROPIC_KEY = ""  # paste sk-ant-...

sf.connect("anthropic", api_key=ANTHROPIC_KEY)

In [ ]:
# Option D — Hugging Face Inference API (open-source / gated models)
# Get a token at https://huggingface.co/settings/tokens
HF_TOKEN = ""  # paste hf_...

sf.connect("huggingface", api_key=HF_TOKEN)

# After connecting, discover which HF models are enabled on this engine:
sf.models.list()

---
## 3 · How IF-Eval is implemented in the engine

ScreamingFace's IF-Eval implementation is designed for reproducibility first.

### Dataset

The 541-prompt Google IFEval dataset is pinned by content hash
(`966cd89545d6b6acfd7638bc708b98261ca58e84`). The engine refuses to run against any
dataset that does not match this hash, so benchmark drift is impossible.

### Verifier

Grading uses a **vendored fork** of the official `instruction_following_eval` verifier —
the same one the original paper authors used. It is pinned into the engine as a
first-class dependency, not a subprocess call to a live package.

### Revision hash

The benchmark exposes a **revision** identifier — a SHA-256 over the dataset content,
verifier source, protocol spec, and result schema. You will see it in the URL4 expression
as a path segment like `/benchmarks/ifeval/1cba769ece27f7ef/`. Any change to any of those
four inputs produces a new revision, and results from different revisions are never mixed.

### Grading protocol

Each case is graded **deterministically** — no model calls, no sampling. The check surface
returns:

- `passed`: `True` / `False` (strict: all constraints must pass)
- `satisfaction`: float in `[0, 1]` (fraction of individual constraints passed)
- `feedback`: plain-text description of which constraints failed and why

The benchmark score is `strict_prompts_passed / 541` — identical to the metric the IFEval
paper uses for its primary leaderboard column.

### Constraint types

IFEval constraints cover ~25 categories. Representative examples:
- `RESPONSE_LANGUAGE` — output must be in a specific language
- `LETTER_FREQUENCY` — a given letter must appear exactly N times
- `RESPONSE_STARTS_WITH` / `RESPONSE_ENDS_WITH`
- `NUMBER_SENTENCES` — exact sentence count
- `JSON_FORMAT` — output must be parseable JSON
- `FORBIDDEN_WORDS` — specific words must not appear

Because the verifier is deterministic and free, the corrective patterns in
ScreamingFace (`CorrectiveLoop`, `SelfCorrective`) can gate on real check results between
rounds at **zero additional model cost** — a property that makes IF-Eval particularly well
suited to studying corrective loop economics.

In [ ]:
# Inspect the benchmark catalog on your local engine
sf.leaderboards.list()

---
## 4 · Caching architecture

The Gateway implements **request-level deduplication** across all model calls. Understanding
this layer is important for interpreting tokenomics measurements.

### Cache key

The cache key is a pure projection of the request content:
```
key = hash(provider, model, messages[], generation_params)
```
Only parameters that affect model output are included. Metadata, request IDs, and
observability fields are excluded, so semantically identical calls always hit the same key.

### Write-through semantics

On a cache miss the Gateway dispatches to the upstream provider, then writes the result via
`set_if_absent` (idempotent — concurrent identical requests converge to one upstream call).
On a hit the response is returned immediately with zero provider latency.

### What this means for ensemble economics

In a `Fusion`, all members receive the same input prompt. If two members use **different
models**, each call is a distinct key and no cache benefit applies within a single run.
Cache savings emerge across **repeated runs** of the same recipe:

- Run 1: all member calls are misses — full provider cost
- Run 2 (same recipe, same benchmark): hits — provider cost ≈ $0

This matters significantly for corrective loop studies: the first pass through IF-Eval
pays full member cost; any re-evaluation of the same recipe (e.g., for ablation, parameter
tuning, or result verification) is served from cache. The `cost_usd` field in `Usage` 
accounts for the actual spend including cache discounts.

### Opting out

To measure raw (uncached) costs, pass `cache_control` to the model params:

```python
model = sf.Model("openrouter/...", params={"cache_control": {"cache_behavior": "bypass"}})
```

### Reading cache statistics

After a run, `result.usage.cache_read_tokens` shows how many tokens were served from
cache rather than billed as `input_tokens`. The `cache` field on each operation in the
detailed accounting records `hits`, `misses`, and `bypasses` per request.

---
## 5 · Building candidates: Solo, Fusion, Pipeline

ScreamingFace has three recipe primitives. All three are nestable and composable.
They compile to a single **URL4 expression** that the Engine executes.

Set up shared parameters first:

In [ ]:
IFEVAL_PARAMS = {"max_tokens": 8192, "temperature": 0.0}

ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)

# Adjust model strings for the provider you connected above.
# OpenRouter: "openrouter/<author>/<model>"
# OpenAI direct: "openai/<model>"
# Anthropic direct: "anthropic/<model>"
# Hugging Face: "huggingface/<org>/<model>" (see sf.models.list() for available slugs)

### 5a · Solo — one model, one pass

The canonical baseline. One model call per case; the raw completion is submitted to the
verifier with no post-processing.

In [ ]:
# Via OpenRouter (covers GPT, Claude, Gemini, open-source)
solo = sf.Model(
    model="openrouter/openai/gpt-5.5",
    prompt=ANSWER_PROMPT,
    params=IFEVAL_PARAMS,
)

# Equivalents for direct providers:
# solo = sf.Model(model="openai/gpt-5.5", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS)
# solo = sf.Model(model="anthropic/claude-sonnet-4-6", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS)

solo

### 5b · Fusion — parallel panel with a synthesizer

All member models receive the same prompt and run **in parallel**. A synthesizer model
then receives all member drafts and produces the final answer in a single pass.
No iteration; the synthesizer runs exactly once regardless of whether member drafts pass
the verifier.

This is the `panel + no-loop` cell in the protocol grid.

In [ ]:
SYNTHESIS_PROMPT = (
    "Produce the single best answer to the original request by combining the panel drafts. "
    "Preserve every instruction and formatting constraint stated in the original prompt. "
    "If drafts disagree on a constraint, follow the one that satisfies the constraint."
)

member_a = sf.Model(
    model="openrouter/openai/gpt-5.5",
    prompt=ANSWER_PROMPT,
    params=IFEVAL_PARAMS,
)
member_b = sf.Model(
    model="openrouter/anthropic/claude-sonnet-4-6",
    prompt=ANSWER_PROMPT,
    params=IFEVAL_PARAMS,
)
synthesizer = sf.Model(
    model="openrouter/anthropic/claude-sonnet-4-6",
    prompt=SYNTHESIS_PROMPT,
    params=IFEVAL_PARAMS,
)

fusion = sf.Fusion(
    members=[member_a, member_b],
    name="gpt_plus_claude",
    synthesizer=synthesizer,
)

fusion

### 5c · Pipeline — ordered sequential stages

A Pipeline passes the output of each stage as input to the next. Unlike Fusion (parallel
fan-out to a synthesizer), Pipeline is strictly sequential: stage 2 can see and build on
stage 1's output.

A natural use case for IF-Eval: a draft model generates a first-pass answer, then a
constraint-checker model revises it to fix any format violations before submission. The
pipeline name is auto-inferred from stage names unless you supply one explicitly.

In [ ]:
REVISE_PROMPT = (
    "You will receive the original instruction and a first-draft answer. "
    "Your job is to revise the draft so that it satisfies every formatting and content "
    "constraint in the original instruction. Do not change correct content — only fix "
    "constraint violations. Output the revised answer only."
)

drafter = sf.Model(
    model="openrouter/openai/gpt-5.5",
    prompt=ANSWER_PROMPT,
    params=IFEVAL_PARAMS,
)
reviser = sf.Model(
    model="openrouter/anthropic/claude-sonnet-4-6",
    prompt=REVISE_PROMPT,
    params=IFEVAL_PARAMS,
)

pipeline = sf.Pipeline(
    stages=[drafter, reviser],
    name="draft_then_revise",
)

pipeline

### How these primitives compose into corrective patterns

The three primitives above are the building blocks for ScreamingFace's corrective loop
recipes, which correspond directly to the patterns studied in the tokenomics paper:

#### `sf.CorrectiveLoop` — panel + corrective (Ens-1 from the paper)

Conceptually: a **Fusion inside a loop**, gated on the check surface.

```
for round in 1..max_rounds:
    drafts = Fusion(members).run(prompt + feedback_from_prev_round)
    best_passing = max(drafts, key=verifier_score)
    if best_passing.passed:
        return best_passing          # submit verbatim — no synthesis step
    feedback = verifier.feedback(drafts)
return best_draft_regardless        # fallback after max_rounds
```

Key properties:
- Members run in **parallel** each round (same wall-clock cost as a Fusion per round)
- If a draft passes the verifier, it is submitted **verbatim** — the judge picks the best
  passing draft, not a synthesized blend
- The verifier call is **free** on IF-Eval (deterministic), so all per-round cost is
  member calls + one judge call
- A first-round pass costs exactly: N member calls + 1 free check

```python
# Ens-1 from the tokenomics paper:
loop = sf.CorrectiveLoop(
    members=[
        sf.Model("openrouter/openai/gpt-5.4-mini", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS),
        sf.Model("openrouter/google/gemini-3.1-flash", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS),
    ],
    judge=sf.Model("openrouter/google/gemini-3.1-flash", params=IFEVAL_PARAMS),
    max_rounds=3,
)
```

#### `sf.SelfCorrective` — solo + corrective

Conceptually: a **Pipeline where the single model re-sits** with its own check feedback
appended to the context.

```
for round in 1..max_rounds:
    draft = model.run(prompt + "\n\n[Feedback from round N-1]: " + feedback)
    if verifier(draft).passed:
        return draft
    feedback = verifier.feedback(draft)
return last_draft
```

Key properties:
- One model call per round (cheapest corrective option)
- Verifier feedback is the study notes — the model learns what it violated
- A first-round pass costs exactly: 1 model call + 1 free check

```python
self_corrective = sf.SelfCorrective(
    sf.Model("openrouter/openai/gpt-5.5", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS),
    max_rounds=3,
)
```

Both patterns are available as first-class recipes. The code blocks above are shown
for conceptual clarity — see `07_ifeval.ipynb` in the examples for runnable versions of
the full protocol grid.

---
## 6 · Run the evaluation

`sf.evaluate` accepts any mix of `sf.Model`, `sf.Fusion`, and `sf.Pipeline` candidates.
It compiles each to a URL4 expression and dispatches to the Engine.

`limit=1` runs a single case — use this to verify your setup before a full run.
Remove `limit` or set it to `None` to run all 541 cases.

In [ ]:
report = sf.evaluate(
    [solo, fusion, pipeline],
    benchmark="ifeval",
    limit=1,         # remove for full 541-case run
    progress=True,
)

report

---
## 7 · Scores and cost accounting

Scores are `strict_prompts_passed / total_cases` — identical to the primary IFEval metric.
For a `limit=1` run the score is 0.0 or 1.0 (binary); a full run gives a fraction.

In [ ]:
for name, result in report.candidates.items():
    print(f"{name:30s}  score={result.score}")

In [ ]:
# Per-candidate token and cost breakdown.
# cache_read_tokens: tokens served from cache (no upstream cost).
# cost_usd: actual spend after cache discounts.
for name, result in report.candidates.items():
    u = result.usage
    print(
        f"{name}:\n"
        f"  input_tokens       = {u.input_tokens}\n"
        f"  output_tokens      = {u.output_tokens}\n"
        f"  cache_read_tokens  = {u.cache_read_tokens}\n"
        f"  cost_usd           = {u.cost_usd}\n"
    )

---
## 8 · URL4 — reproducibility

Every candidate compiles to a **url4** expression that encodes the complete recipe:
member model bindings, prompts, generation parameters, benchmark revision, and protocol
version. The expression is self-contained — anyone with the same url4 and access to the
same provider can reproduce the run and get the same score (subject to model determinism;
`temperature=0.0` helps).

The leaderboard stores url4 alongside every submitted score, making every entry on the
public board inspectable and re-runnable.

In [ ]:
fusion_result = report.candidates["gpt_plus_claude"]

print("Fusion url4:")
print(fusion_result.url4)
print()

# The benchmark revision appears as a path segment:
# /benchmarks/ifeval/<revision>/cases*
# That revision is the SHA-256 over dataset + verifier + protocol + schema.
# Results from different revisions are never compared.

---
## 9 · Submit to the public leaderboard (optional)

Set `PUBLISH = True` to publish your result. The leaderboard deduplicates on run ID, so
re-running the same recipe does not create duplicate entries. Results submitted from a
local engine are indistinguishable from platform-mode results — the score and url4 are
what get stored, not where the engine ran.

In [ ]:
PUBLISH = False

submission = sf.leaderboards.submit(fusion_result) if PUBLISH else None
submission or "Set PUBLISH = True to submit."

In [ ]:
sf.leaderboards.get("ifeval", top=10)

---
## 10 · Teardown

In [ ]:
sf.close()
!screamingface down

---
### Next steps

- **Full protocol grid** (Solo / Fusion / SelfCorrective / CorrectiveLoop across IF-Eval):
  `packages/screamingface/examples/07_ifeval.ipynb`
- **Platform mode** (hosted engine, zero local setup): `ScreamingFace_Platform.ipynb`
- **Client API reference**: https://docs.screamingface.ai
- **Questions or feedback**: reach out to the team.